# 00 — Upstream cohort construction

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 00 — Bluesky ADRD cohort construction (reproducibility version)

**Purpose.** Reconstruct the upstream cohort-construction workflow used before the main revision pipeline.
This notebook is a cleaned, path-portable version of the historical `data_filtering.ipynb` workflow.
It preserves the **analytical decisions actually used** while making the execution order and handoff files explicit.

The final output of this notebook is:

`matched_subset_exact_fuzzy_no_political_5_7_new.parquet`

That file is the direct input to `01_main_revision_pipeline_NO_BRAIN_FOG_FINAL_CORRECTED.ipynb`.

## Important reproducibility notes

- The raw Bluesky archive is **not redistributed** here. Obtain it from the source dataset described in the manuscript/repository README.
- The original workflow contained a manual review step in which keywords were assigned one of three matching rules: `exact = 1`, `exact = 0`, or blank/NA. The reviewed rule file must therefore be published with the repository.
- In this historical workflow, **"fuzzy" means substring matching**, not edit-distance or probabilistic fuzzy matching.
- The political/noise exclusion list below is reproduced exactly from the historical notebook. It is not re-designed here.
- Exploratory diagnostics from the historical notebook (sentiment summaries, daily counts, co-occurrence exploration, etc.) are intentionally omitted because they were not required to create the handoff parquet used by Notebook 01.
- Heavy steps are disabled by default. Set `RUN_HEAVY_STEPS = True` only when intentionally reconstructing the cohort from the raw archive.


## Expected historical checkpoints

These counts are preserved from the saved outputs of the original notebook and are provided as **reproducibility checks**, not recomputed values in this cleaned notebook:

| Stage | Historical row count |
|---|---:|
| Broad substring candidate set | 1,126,036 |
| After reviewed exact/fuzzy/skip rules | 422,011 |
| After political/noise substring exclusions | 388,104 |

The later revision notebook re-filters the final 388,104-row handoff parquet using the refined 17-keyword specification and produces the final revised analytic cohort.


## 1. Configuration

Recommended repository layout:

- `data/raw/` — raw downloaded source archive; not committed to GitHub
- `data/intermediate/` — large intermediate parquet files; normally not committed
- `data/processed/` — final upstream handoff parquet; normally not committed if large
- `config/ADRD_keywords_4_1.xlsx` — original candidate keyword workbook
- `config/keyword_rules_reviewed.xlsx` — reviewed matching-rule file with columns `keyword`, `frequency`, and `exact`
- `notebooks/` — analysis notebooks

If your local folder structure differs, edit only the paths in the next cell.


In [ ]:
from pathlib import Path
import os
import re
import tarfile
import time
from collections import Counter

import numpy as np
import orjson
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# -----------------------------------------------------------------------------
# Project paths
# -----------------------------------------------------------------------------
# If this notebook lives in notebooks/, parent directory is the repository root.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_DIR = PROJECT_ROOT / "config"

RAW_POSTS_TAR = RAW_DIR / "user_posts.tar.gz"
ENGLISH_PARQUET_DIR = INTERMEDIATE_DIR / "filtered_eng_parquet"
BROAD_MATCH_CHUNK_DIR = INTERMEDIATE_DIR / "filtered_eng_parquet_4_23"
BROAD_MATCH_COMBINED = INTERMEDIATE_DIR / "matched_combined_4_23.parquet"

BROAD_KEYWORD_FILE = CONFIG_DIR / "broad_keywords_minimized.txt"
# This file represents the manual keyword-rule review performed in the original workflow.
# Required columns: keyword, frequency, exact
# exact = 1 -> word-boundary exact matching
# exact = 0 -> skip keyword
# exact blank/NA -> substring matching
REVIEWED_KEYWORD_RULES_FILE = CONFIG_DIR / "keyword_rules_reviewed.xlsx"

EXACT_FUZZY_OUTPUT = INTERMEDIATE_DIR / "matched_subset_exact_fuzzy_no_exclude_5_7.parquet"
FINAL_HANDOFF_PARQUET = PROCESSED_DIR / "matched_subset_exact_fuzzy_no_political_5_7_new.parquet"

# Safety: this pipeline is expensive and may create very large files.
RUN_HEAVY_STEPS = False
OVERWRITE_EXISTING = False

for folder in [INTERMEDIATE_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_POSTS_TAR:", RAW_POSTS_TAR)
print("FINAL_HANDOFF_PARQUET:", FINAL_HANDOFF_PARQUET)
print("RUN_HEAVY_STEPS:", RUN_HEAVY_STEPS)


## 2. Optional source-data inspection

This reproduces the original notebook's schema/language checks. These cells are descriptive only and do not change the cohort definition.


In [ ]:
def preview_raw_schema(tar_path, sample_size=500):
    """Read up to `sample_size` rows from the first JSONL member and show fields."""
    rows = []
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar:
            if not member.name.endswith(".jsonl"):
                continue
            f = tar.extractfile(member)
            if f is None:
                continue
            for _ in range(sample_size):
                line = f.readline()
                if not line:
                    break
                rows.append(orjson.loads(line))
            break
    sample = pd.DataFrame(rows)
    print(sample.head())
    print("\nColumns:", sample.columns.tolist())
    return sample

# Uncomment for source inspection only:
# sample_df = preview_raw_schema(RAW_POSTS_TAR)


## 3. Filter English-language posts

The historical rule is preserved: keep a post when `langs` is a list containing `"eng"`.
The output is written as a month-partitioned Parquet dataset.

Unlike the historical notebook, this cleaned version includes the **actual function call** inside a guarded execution cell so the step is explicit and reproducible.


In [ ]:
def extract_english_posts_to_parquet(
    tar_path,
    output_folder,
    batch_size=10_000,
    overwrite=False,
):
    output_folder = Path(output_folder)

    if output_folder.exists() and any(output_folder.rglob("*.parquet")) and not overwrite:
        raise FileExistsError(
            f"English parquet output already exists: {output_folder}\n"
            "Set OVERWRITE_EXISTING=True only if you intentionally want to rebuild it."
        )

    output_folder.mkdir(parents=True, exist_ok=True)
    batch_data = []
    error_count = 0
    n_kept = 0
    start = time.time()

    with tarfile.open(tar_path, "r:gz") as tar:
        progress = tqdm(desc="Reading JSONL members", unit="file")

        for member in tar:
            if not member.name.endswith(".jsonl"):
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            for line in f:
                try:
                    record = orjson.loads(line)
                except orjson.JSONDecodeError:
                    error_count += 1
                    continue

                langs = record.get("langs") or []
                if isinstance(langs, list) and "eng" in langs:
                    record["year_month"] = str(record.get("date", "unknown"))[:7]
                    batch_data.append(record)
                    n_kept += 1

                if len(batch_data) >= batch_size:
                    table = pa.Table.from_pandas(pd.DataFrame(batch_data), preserve_index=False)
                    pq.write_to_dataset(
                        table,
                        root_path=str(output_folder),
                        partition_cols=["year_month"],
                        existing_data_behavior="overwrite_or_ignore",
                    )
                    batch_data = []

            progress.update(1)

        if batch_data:
            table = pa.Table.from_pandas(pd.DataFrame(batch_data), preserve_index=False)
            pq.write_to_dataset(
                table,
                root_path=str(output_folder),
                partition_cols=["year_month"],
                existing_data_behavior="overwrite_or_ignore",
            )

        progress.close()

    print(f"English-language rows written: {n_kept:,}")
    print(f"Corrupted JSON lines skipped: {error_count:,}")
    print(f"Elapsed seconds: {time.time() - start:.1f}")


if RUN_HEAVY_STEPS:
    extract_english_posts_to_parquet(
        RAW_POSTS_TAR,
        ENGLISH_PARQUET_DIR,
        overwrite=OVERWRITE_EXISTING,
    )
else:
    print("SKIPPED heavy step. Set RUN_HEAVY_STEPS=True to rebuild English-language parquet files.")


## 4. Build the initial broad keyword list

This function preserves the historical logic exactly:

1. Read the `Keyword` column.
2. Split comma-separated entries and lowercase them.
3. Remove duplicates.
4. If one keyword is contained inside another, remove the **longer** term.

That fourth rule is a historical analytical decision. It is preserved here rather than silently redesigned.


In [ ]:
# -----------------------------------------------------------------------------
# Historical broad candidate keyword list
# -----------------------------------------------------------------------------
# The original executed notebook read a larger workbook and minimized overlapping
# phrases before broad retrieval. For public reproducibility, the exact minimized
# list actually used by that executed notebook is stored directly in config/.

def load_keyword_list(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing keyword list: {path}")
    return [
        line.strip().lower()
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]

broad_keywords = load_keyword_list(BROAD_KEYWORD_FILE)
print(f"Broad minimized keyword count: {len(broad_keywords)}")


## 5. Broad candidate retrieval by substring matching

The historical candidate-retrieval stage used **case-insensitive substring matching**: a post was retained if *any* broad keyword occurred anywhere in its text.

The saved output from the original run contained **1,126,036 rows**.


In [ ]:
def collect_parquet_files(folder):
    return sorted(str(p) for p in Path(folder).rglob("*.parquet"))


def match_broad_keywords_in_file(parquet_file, keyword_list):
    try:
        table = ds.dataset(parquet_file, format="parquet").to_table()
        frame = table.to_pandas()
        text_lower = frame["text"].astype(str).str.lower()
        mask = text_lower.apply(lambda x: any(kw in x for kw in keyword_list))
        return frame.loc[mask]
    except Exception as exc:
        print(f"WARNING: skipped {parquet_file}: {exc}")
        return None


def broad_candidate_retrieval(
    english_parquet_dir,
    keyword_list,
    chunk_output_dir,
    combined_output,
    chunk_size=5_000,
    max_threads=16,
    overwrite=False,
):
    chunk_output_dir = Path(chunk_output_dir)
    combined_output = Path(combined_output)

    if combined_output.exists() and not overwrite:
        raise FileExistsError(
            f"Combined candidate file already exists: {combined_output}\n"
            "Set OVERWRITE_EXISTING=True only for an intentional rebuild."
        )

    chunk_output_dir.mkdir(parents=True, exist_ok=True)
    parquet_files = collect_parquet_files(english_parquet_dir)
    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found under {english_parquet_dir}")

    from concurrent.futures import ThreadPoolExecutor

    buffered = []
    chunk_index = 0
    total_matched = 0
    written_chunks = []

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        iterator = executor.map(
            lambda p: match_broad_keywords_in_file(p, keyword_list),
            parquet_files,
        )
        for matched in tqdm(iterator, total=len(parquet_files), desc="Broad keyword matching"):
            if matched is None or matched.empty:
                continue

            buffered.append(matched)
            total_matched += len(matched)

            if sum(len(x) for x in buffered) >= chunk_size:
                chunk_df = pd.concat(buffered, ignore_index=True)
                chunk_path = chunk_output_dir / f"matched_chunk_{chunk_index:04d}.parquet"
                pq.write_table(pa.Table.from_pandas(chunk_df, preserve_index=False), chunk_path)
                written_chunks.append(chunk_path)
                buffered = []
                chunk_index += 1

    if buffered:
        chunk_df = pd.concat(buffered, ignore_index=True)
        chunk_path = chunk_output_dir / f"matched_chunk_{chunk_index:04d}.parquet"
        pq.write_table(pa.Table.from_pandas(chunk_df, preserve_index=False), chunk_path)
        written_chunks.append(chunk_path)

    # Combine only the chunks written by this run.
    combined_df = pd.concat([pd.read_parquet(p) for p in written_chunks], ignore_index=True)
    pq.write_table(pa.Table.from_pandas(combined_df, preserve_index=False), combined_output)

    print(f"Broad candidate rows: {len(combined_df):,}")
    print("Historical checkpoint: 1,126,036")
    print("Combined output:", combined_output)
    return combined_df


if RUN_HEAVY_STEPS:
    if broad_keywords is None:
        raise RuntimeError("Broad keywords are unavailable; provide the initial keyword workbook first.")
    broad_df = broad_candidate_retrieval(
        ENGLISH_PARQUET_DIR,
        broad_keywords,
        BROAD_MATCH_CHUNK_DIR,
        BROAD_MATCH_COMBINED,
        overwrite=OVERWRITE_EXISTING,
    )
else:
    print("SKIPPED heavy step. Historical broad-candidate checkpoint: 1,126,036 rows.")


## 6. Keyword-frequency review and matching-rule configuration

The original workflow next counted keyword occurrences within the broad candidate set and then **manually reviewed the keyword table**.
The reviewed table added an `exact` field:

- `exact = 1` → retain only word-boundary exact matches
- `exact = 0` → exclude/skip this keyword
- `exact` blank/NA → retain by substring matching

For a public repository, do **not** hide this step. Publish the reviewed rule file as `config/keyword_rules_reviewed.xlsx` (or a CSV with equivalent columns).


In [ ]:
def build_keyword_frequency_table(candidate_parquet, keyword_list, output_file=None):
    frame = pq.read_table(candidate_parquet).to_pandas()
    text_lower = frame["text"].astype(str).str.lower()

    escaped = [re.escape(kw) for kw in keyword_list]
    regex = re.compile("(" + "|".join(escaped) + ")")

    all_matches = []
    for text in tqdm(text_lower, desc="Counting broad keyword matches"):
        all_matches.extend(regex.findall(text))

    counts = Counter(all_matches)
    freq = (
        pd.DataFrame(counts.items(), columns=["keyword", "frequency"])
        .sort_values("frequency", ascending=False)
        .reset_index(drop=True)
    )

    if output_file is not None:
        output_file = Path(output_file)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        freq.to_excel(output_file, index=False)
        print("Saved frequency table for review:", output_file)

    return freq


# Optional reconstruction of the pre-review frequency table.
# This is not needed when the reviewed rule file is already published in config/.
if RUN_HEAVY_STEPS and BROAD_MATCH_COMBINED.exists() and broad_keywords is not None:
    review_input = INTERMEDIATE_DIR / "keyword_frequency_4_23_for_review.xlsx"
    keyword_frequency = build_keyword_frequency_table(
        BROAD_MATCH_COMBINED,
        broad_keywords,
        review_input,
    )
    print(
        "Manual reproducibility step: review this table and encode the historical exact/fuzzy/skip rules. "
        "For the paper repository, use the already-reviewed rule file rather than re-adjudicating keywords."
    )


In [ ]:
def load_reviewed_keyword_rules(rule_file):
    rules = pd.read_excel(rule_file, engine="openpyxl")
    rules.columns = rules.columns.str.strip().str.lower()

    required = {"keyword", "exact"}
    missing = required - set(rules.columns)
    if missing:
        raise KeyError(
            f"Reviewed rule file is missing columns: {sorted(missing)}. "
            "The original workflow requires an 'exact' column."
        )

    rules["keyword"] = rules["keyword"].astype(str).str.strip().str.lower()

    exact_keywords = rules.loc[rules["exact"] == 1, "keyword"].dropna().tolist()
    skip_keywords = rules.loc[rules["exact"] == 0, "keyword"].dropna().tolist()
    fuzzy_keywords = rules.loc[rules["exact"].isna(), "keyword"].dropna().tolist()

    print(f"Exact keywords: {len(exact_keywords)}")
    print(f"Substring ('fuzzy') keywords: {len(fuzzy_keywords)}")
    print(f"Skipped keywords: {len(skip_keywords)}")
    return rules, exact_keywords, fuzzy_keywords, skip_keywords


if REVIEWED_KEYWORD_RULES_FILE.exists():
    keyword_rules, exact_keywords, fuzzy_keywords, skip_keywords = load_reviewed_keyword_rules(
        REVIEWED_KEYWORD_RULES_FILE
    )
else:
    keyword_rules = exact_keywords = fuzzy_keywords = skip_keywords = None
    print("Reviewed keyword-rule file not found:", REVIEWED_KEYWORD_RULES_FILE)
    print("Publish the historical reviewed rule file in config/ before claiming full reproducibility.")


## 7. Apply the reviewed exact/fuzzy/skip rules

This stage reproduces the historical rule exactly:

- exact keywords use word boundaries;
- blank/NA keywords use substring matching;
- `exact = 0` keywords do not participate in retrieval.

Historical checkpoint after this stage: **422,011 rows**.


In [ ]:
def apply_reviewed_keyword_rules(
    candidate_parquet,
    exact_keywords,
    fuzzy_keywords,
    output_file,
    n_jobs=8,
    overwrite=False,
):
    output_file = Path(output_file)
    if output_file.exists() and not overwrite:
        raise FileExistsError(
            f"Output already exists: {output_file}\n"
            "Set OVERWRITE_EXISTING=True only for an intentional rebuild."
        )

    exact_pattern = (
        r"\b(" + "|".join(re.escape(kw) for kw in exact_keywords) + r")\b"
        if exact_keywords else r"$^"
    )
    fuzzy_pattern = (
        "(" + "|".join(re.escape(kw) for kw in fuzzy_keywords) + ")"
        if fuzzy_keywords else r"$^"
    )
    regex_exact = re.compile(exact_pattern)
    regex_fuzzy = re.compile(fuzzy_pattern)

    frame = pq.read_table(candidate_parquet).to_pandas()

    def match_text(text):
        text = str(text).lower()
        exact_matches = regex_exact.findall(text)
        fuzzy_matches = regex_fuzzy.findall(text)
        return list(set(exact_matches + fuzzy_matches))

    def process_chunk(chunk):
        return chunk["text"].apply(match_text)

    chunks = np.array_split(frame, n_jobs)
    matched_chunks = Parallel(n_jobs=n_jobs)(
        delayed(process_chunk)(chunk) for chunk in chunks
    )

    frame["matched_keywords"] = pd.concat(matched_chunks).sort_index()
    matched = frame.loc[frame["matched_keywords"].apply(len) > 0].copy()

    pq.write_table(pa.Table.from_pandas(matched, preserve_index=False), output_file)

    print(f"Rows after reviewed exact/fuzzy/skip rules: {len(matched):,}")
    print("Historical checkpoint: 422,011")
    print("Saved:", output_file)
    return matched


if RUN_HEAVY_STEPS:
    if not BROAD_MATCH_COMBINED.exists():
        raise FileNotFoundError(BROAD_MATCH_COMBINED)
    if exact_keywords is None or fuzzy_keywords is None:
        raise RuntimeError("Reviewed keyword rules are required before this step.")

    exact_fuzzy_df = apply_reviewed_keyword_rules(
        BROAD_MATCH_COMBINED,
        exact_keywords,
        fuzzy_keywords,
        EXACT_FUZZY_OUTPUT,
        overwrite=OVERWRITE_EXISTING,
    )
else:
    print("SKIPPED heavy step. Historical exact/fuzzy/skip checkpoint: 422,011 rows.")


## 8. Apply the historical political/noise exclusions

The following terms and substring logic are reproduced exactly from the historical notebook.
A row is removed if its lowercased text contains **any** listed substring.

This is a cohort-construction decision and must remain visible in the public repository.


In [ ]:
POLITICAL_EXCLUSION_TERMS = [
    "biden", "trump", "president", "whitehouse", "congress", "democrat", "republic",
    "government", "russia", "israel", "palestine", "ceasefire", "gaza", "palestinian",
    "hamas", "zionist", "election", "politic", "putin", "ukra", "war", "nazi",
]

print("Historical political/noise exclusion terms:")
print(POLITICAL_EXCLUSION_TERMS)


In [ ]:
def apply_political_exclusions(
    input_parquet,
    output_parquet,
    exclusion_terms,
    overwrite=False,
):
    input_parquet = Path(input_parquet)
    output_parquet = Path(output_parquet)

    if output_parquet.exists() and not overwrite:
        raise FileExistsError(
            f"Final handoff output already exists: {output_parquet}\n"
            "Set OVERWRITE_EXISTING=True only for an intentional rebuild."
        )

    frame = pq.read_table(input_parquet).to_pandas()
    text_lower = frame["text"].astype(str).str.lower()

    keep_mask = text_lower.apply(
        lambda x: not any(term in x for term in exclusion_terms)
    )
    final_df = frame.loc[keep_mask].copy()

    output_parquet.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(pa.Table.from_pandas(final_df, preserve_index=False), output_parquet)

    print(f"Rows after political/noise exclusion: {len(final_df):,}")
    print("Historical checkpoint: 388,104")
    print("Final handoff parquet:", output_parquet)
    return final_df


if RUN_HEAVY_STEPS:
    if not EXACT_FUZZY_OUTPUT.exists():
        raise FileNotFoundError(EXACT_FUZZY_OUTPUT)
    final_upstream_df = apply_political_exclusions(
        EXACT_FUZZY_OUTPUT,
        FINAL_HANDOFF_PARQUET,
        POLITICAL_EXCLUSION_TERMS,
        overwrite=OVERWRITE_EXISTING,
    )
else:
    print("SKIPPED heavy step. Historical final upstream checkpoint: 388,104 rows.")


## 9. Reproducibility checks and handoff to Notebook 01

The final upstream file name is intentionally unchanged because Notebook 01 reads this exact handoff file and then applies the refined final 17-keyword revision specification.

The expected file name is:

`matched_subset_exact_fuzzy_no_political_5_7_new.parquet`

If reconstructing the pipeline, compare the row counts above with the historical checkpoints. Small implementation changes should **not** be introduced merely to make the counts match; any discrepancy should be investigated and documented.


In [ ]:
EXPECTED_HISTORICAL_COUNTS = {
    "broad_candidate_rows": 1_126_036,
    "exact_fuzzy_rows": 422_011,
    "final_upstream_rows": 388_104,
}

print("Expected historical checkpoints:")
for key, value in EXPECTED_HISTORICAL_COUNTS.items():
    print(f"  {key}: {value:,}")

print("\nNotebook 01 handoff filename:")
print(FINAL_HANDOFF_PARQUET.name)

if FINAL_HANDOFF_PARQUET.exists():
    final_count = pq.read_metadata(FINAL_HANDOFF_PARQUET).num_rows
    print(f"Existing local final handoff rows: {final_count:,}")
    if final_count != EXPECTED_HISTORICAL_COUNTS["final_upstream_rows"]:
        print("WARNING: local handoff row count differs from the preserved historical checkpoint.")
else:
    print("Final handoff parquet is not present locally in this repository layout (expected for GitHub code-only release).")


## 10. What was intentionally removed from the historical notebook

The original notebook also contained exploratory or diagnostic code for:

- sentiment summaries;
- post-length/user-count summaries;
- keyword co-occurrence tables;
- daily English-post counts;
- commented-out experimental filters.

Those cells were not required to generate the upstream handoff parquet consumed by Notebook 01, so they are omitted from this reproducibility notebook. The unmodified historical notebook can be retained privately or archived separately for provenance.

## Recommended public-repository practice

Keep both:

1. this cleaned notebook as the executable/reviewable cohort-construction workflow; and
2. the original historical notebook in an `archive/` folder only if you want to provide full provenance.

Do not describe this cleaned notebook as a newly re-run analysis unless you actually rerun it. A precise repository note is: **"Code reorganized for reproducibility and path portability without changing the cohort-construction rules used in the study."**
